# Week 2: Embedded DSL for Scenario-Aware Attention

This notebook extends Week 1 by introducing a small Python-embedded domain-specific language (DSL) for expressing attention workloads declaratively. Rather than optimizing attention execution yet, the goal of this week is to define a programmable interface that can represent scenario-specific attention behavior and lower it into a correct baseline PyTorch implementation.

The motivation comes from the project feedback that general-purpose transformer optimization is difficult to distinguish from existing state-of-the-art solutions. In response, this project focuses on application scenarios where customized attention execution may provide practical value, including multi-tenant serving, streaming inference, and long-context systems.

This week therefore focuses on expressiveness and correctness. The DSL will allow attention computations to be described using high-level workload metadata, and the resulting specification will be translated into standard baseline attention operations. Later weeks will build on this interface to introduce memory-efficient tiled transformations.

## Week 2 Objectives

The goals of this notebook are:

1. Reuse the baseline attention functions from Week 1.
2. Define a lightweight Python-embedded DSL for attention workloads.
3. Represent application-specific execution settings declaratively.
4. Lower DSL specifications into baseline PyTorch attention computations.
5. Verify correctness by comparing DSL-lowered outputs to direct reference implementations.

This notebook does not yet implement FlashAttention-style tiling. Instead, it establishes the programmable interface that later transformations will target.

In [ ]:
import math
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


## Baseline Attention Functions

The following functions are reused from Week 1. They serve as the reference implementations that the DSL will lower into during this stage of the project.

In [ ]:
def baseline_attention(Q, K, V, causal=False):
    """
    Standard scaled dot-product attention.
    Supports optional causal masking.
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if causal:
        seq_len_q = Q.size(-2)
        seq_len_k = K.size(-2)
        mask = torch.triu(
            torch.ones(seq_len_q, seq_len_k, device=Q.device, dtype=torch.bool),
            diagonal=1
        )
        scores = scores.masked_fill(mask, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    return torch.matmul(weights, V)

In [ ]:
def streaming_attention(Q_seq, K_seq, V_seq):
    """
    Simple streaming-style reference attention.
    At time step t, attention is computed only over the prefix up to t.
    """
    outputs = []

    for t in range(Q_seq.size(1)):
        Q_t = Q_seq[:, t:t+1, :]
        K_t = K_seq[:, :t+1, :]
        V_t = V_seq[:, :t+1, :]
        out_t = baseline_attention(Q_t, K_t, V_t, causal=False)
        outputs.append(out_t)

    return torch.cat(outputs, dim=1)

## Quick Baseline Sanity Check

Before building the DSL, we verify that the baseline attention function runs correctly on a small example.

In [ ]:
B, N, D = 2, 8, 16

Q = torch.randn(B, N, D, device=device)
K = torch.randn(B, N, D, device=device)
V = torch.randn(B, N, D, device=device)

baseline_output = baseline_attention(Q, K, V)
print("Baseline output shape:", baseline_output.shape)

Baseline output shape: torch.Size([2, 8, 16])


## DSL Design

The embedded DSL represents an attention workload as a Python object with both tensor inputs and scenario-specific metadata. This keeps the interface lightweight while still making execution intent explicit.

The DSL includes:
- the attention inputs `Q`, `K`, and `V`
- a workload name
- a target application scenario
- optional execution settings such as causal masking, tile size, streaming mode, variable-length handling, and KV-cache intent

At this stage, these settings are mainly descriptive. In later weeks, they will guide transformed execution.

In [ ]:
class AttentionSpec:
    """
    Lightweight DSL object for declarative attention specification.
    """
    def __init__(
        self,
        Q,
        K,
        V,
        name="attention_op",
        scenario="generic",
        causal=False,
        tile_size=None,
        streaming=False,
        variable_length=False,
        cache_kv=False
    ):
        self.Q = Q
        self.K = K
        self.V = V
        self.name = name
        self.scenario = scenario
        self.causal = causal
        self.tile_size = tile_size
        self.streaming = streaming
        self.variable_length = variable_length
        self.cache_kv = cache_kv

    def summary(self):
        return {
            "name": self.name,
            "scenario": self.scenario,
            "causal": self.causal,
            "tile_size": self.tile_size,
            "streaming": self.streaming,
            "variable_length": self.variable_length,
            "cache_kv": self.cache_kv,
            "Q_shape": tuple(self.Q.shape),
            "K_shape": tuple(self.K.shape),
            "V_shape": tuple(self.V.shape),
        }

## Example DSL Specification

The following cell creates a simple generic attention specification and displays its metadata.

In [ ]:
generic_spec = AttentionSpec(
    Q=Q,
    K=K,
    V=V,
    name="generic_attention",
    scenario="generic",
    causal=False,
    tile_size=None,
    streaming=False,
    variable_length=False,
    cache_kv=False
)

pd.DataFrame([generic_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,generic_attention,generic,False,None,False,False,False,"(2, 8, 16)","(2, 8, 16)","(2, 8, 16)"


## Lowering the DSL to Baseline Execution

The next step is to translate a DSL specification into an executable baseline attention computation.

For Week 2, the lowering logic is intentionally simple:
- if the specification is marked as streaming, use the streaming reference function
- otherwise, use the standard baseline attention function
- preserve the causal flag for standard attention

This gives the project a compiler-like structure without introducing optimization yet.

In [ ]:
def lower_to_baseline(spec):
    """
    Lower a DSL attention specification to a baseline reference implementation.
    """
    if spec.streaming:
        return streaming_attention(spec.Q, spec.K, spec.V)
    else:
        return baseline_attention(spec.Q, spec.K, spec.V, causal=spec.causal)

## Correctness Check for Generic Attention

The DSL-lowered result should match the direct baseline attention implementation.

In [ ]:
dsl_output = lower_to_baseline(generic_spec)
ref_output = baseline_attention(Q, K, V)

max_diff = (dsl_output - ref_output).abs().max().item()
print("Maximum absolute difference:", max_diff)
print("Outputs match:", torch.allclose(dsl_output, ref_output, atol=1e-6))

Maximum absolute difference: 0.0
Outputs match: True


## Scenario 1: Multi-Tenant LLM Serving

In a multi-tenant serving environment, requests can have different sequence lengths and may require variable-length handling. Even though this notebook does not yet optimize execution, the DSL can already represent this workload explicitly.

This is one example of how the DSL provides a more informative abstraction than a raw baseline attention call.

In [ ]:
Q_mt = torch.randn(1, 128, 64, device=device)
K_mt = torch.randn(1, 128, 64, device=device)
V_mt = torch.randn(1, 128, 64, device=device)

multitenant_spec = AttentionSpec(
    Q=Q_mt,
    K=K_mt,
    V=V_mt,
    name="multitenant_request",
    scenario="multi_tenant",
    causal=False,
    tile_size=64,
    streaming=False,
    variable_length=True,
    cache_kv=False
)

pd.DataFrame([multitenant_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,multitenant_request,multi_tenant,False,64,False,True,False,"(1, 128, 64)","(1, 128, 64)","(1, 128, 64)"


In [ ]:
mt_dsl_output = lower_to_baseline(multitenant_spec)
mt_ref_output = baseline_attention(Q_mt, K_mt, V_mt)

mt_max_diff = (mt_dsl_output - mt_ref_output).abs().max().item()
print("Multi-tenant max difference:", mt_max_diff)
print("Outputs match:", torch.allclose(mt_dsl_output, mt_ref_output, atol=1e-6))

Multi-tenant max difference: 0.0
Outputs match: True


## Scenario 2: Streaming Inference

For streaming inference, tokens arrive incrementally and attention is applied over a growing prefix. The DSL captures this execution intent through the `streaming=True` setting.

At this stage, lowering routes the specification to the streaming reference implementation.

In [ ]:
Q_stream = torch.randn(1, 32, 64, device=device)
K_stream = torch.randn(1, 32, 64, device=device)
V_stream = torch.randn(1, 32, 64, device=device)

streaming_spec = AttentionSpec(
    Q=Q_stream,
    K=K_stream,
    V=V_stream,
    name="streaming_request",
    scenario="streaming",
    causal=False,
    tile_size=16,
    streaming=True,
    variable_length=False,
    cache_kv=True
)

pd.DataFrame([streaming_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,streaming_request,streaming,False,16,True,False,True,"(1, 32, 64)","(1, 32, 64)","(1, 32, 64)"


In [ ]:
stream_dsl_output = lower_to_baseline(streaming_spec)
stream_ref_output = streaming_attention(Q_stream, K_stream, V_stream)

stream_max_diff = (stream_dsl_output - stream_ref_output).abs().max().item()
print("Streaming max difference:", stream_max_diff)
print("Outputs match:", torch.allclose(stream_dsl_output, stream_ref_output, atol=1e-6))

Streaming max difference: 0.0
Outputs match: True


## Scenario 3: Long-Context Systems

Long-context workloads are especially relevant because they stress the quadratic memory behavior of standard attention. In the DSL, this scenario can be represented directly, along with a tile size that later transformations may use to lower memory usage.

For now, the specification still lowers to the standard baseline implementation.

In [ ]:
Q_long = torch.randn(1, 256, 64, device=device)
K_long = torch.randn(1, 256, 64, device=device)
V_long = torch.randn(1, 256, 64, device=device)

long_context_spec = AttentionSpec(
    Q=Q_long,
    K=K_long,
    V=V_long,
    name="long_context_request",
    scenario="long_context",
    causal=True,
    tile_size=64,
    streaming=False,
    variable_length=False,
    cache_kv=False
)

pd.DataFrame([long_context_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,long_context_request,long_context,True,64,False,False,False,"(1, 256, 64)","(1, 256, 64)","(1, 256, 64)"


In [ ]:
long_dsl_output = lower_to_baseline(long_context_spec)
long_ref_output = baseline_attention(Q_long, K_long, V_long, causal=True)

long_max_diff = (long_dsl_output - long_ref_output).abs().max().item()
print("Long-context max difference:", long_max_diff)
print("Outputs match:", torch.allclose(long_dsl_output, long_ref_output, atol=1e-6))

Long-context max difference: 0.0
Outputs match: True


## Comparing the Scenario Specifications

One benefit of the DSL is that scenario-specific execution intent becomes explicit and inspectable. The same baseline attention computation can now be described differently depending on the intended workload.

In [ ]:
scenario_df = pd.DataFrame([
    multitenant_spec.summary(),
    streaming_spec.summary(),
    long_context_spec.summary()
])

scenario_df

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,multitenant_request,multi_tenant,False,64,False,True,False,"(1, 128, 64)","(1, 128, 64)","(1, 128, 64)"
1,streaming_request,streaming,False,16,True,False,True,"(1, 32, 64)","(1, 32, 64)","(1, 32, 64)"
2,long_context_request,long_context,True,64,False,False,False,"(1, 256, 64)","(1, 256, 64)","(1, 256, 64)"


## Helper: Batch Construction from Scenario Metadata

To make the DSL slightly more realistic, we can add a helper function that creates an `AttentionSpec` from high-level scenario settings. This keeps the user-facing interface compact and makes later compiler-style transformations easier to organize.

In [ ]:
def make_attention_spec(
    seq_len,
    d_model,
    batch_size=1,
    scenario="generic",
    causal=False,
    tile_size=None,
    streaming=False,
    variable_length=False,
    cache_kv=False,
    name="attention_op"
):
    Q = torch.randn(batch_size, seq_len, d_model, device=device)
    K = torch.randn(batch_size, seq_len, d_model, device=device)
    V = torch.randn(batch_size, seq_len, d_model, device=device)

    return AttentionSpec(
        Q=Q,
        K=K,
        V=V,
        name=name,
        scenario=scenario,
        causal=causal,
        tile_size=tile_size,
        streaming=streaming,
        variable_length=variable_length,
        cache_kv=cache_kv
    )

In [ ]:
helper_spec = make_attention_spec(
    seq_len=64,
    d_model=32,
    batch_size=2,
    scenario="multi_tenant",
    causal=False,
    tile_size=32,
    streaming=False,
    variable_length=True,
    cache_kv=False,
    name="helper_generated_spec"
)

pd.DataFrame([helper_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,helper_generated_spec,multi_tenant,False,32,False,True,False,"(2, 64, 32)","(2, 64, 32)","(2, 64, 32)"


## Final Correctness Summary

The following table summarizes whether each DSL specification lowers correctly to its corresponding baseline implementation.

In [ ]:
results_df = pd.DataFrame([
    {
        "scenario": "generic",
        "max_abs_diff": max_diff,
        "matches_reference": torch.allclose(dsl_output, ref_output, atol=1e-6)
    },
    {
        "scenario": "multi_tenant",
        "max_abs_diff": mt_max_diff,
        "matches_reference": torch.allclose(mt_dsl_output, mt_ref_output, atol=1e-6)
    },
    {
        "scenario": "streaming",
        "max_abs_diff": stream_max_diff,
        "matches_reference": torch.allclose(stream_dsl_output, stream_ref_output, atol=1e-6)
    },
    {
        "scenario": "long_context",
        "max_abs_diff": long_max_diff,
        "matches_reference": torch.allclose(long_dsl_output, long_ref_output, atol=1e-6)
    }
])

results_df

,scenario,max_abs_diff,matches_reference
0,generic,0.0,True
1,multi_tenant,0.0,True
2,streaming,0.0,True
3,long_context,0.0,True


## Week 2 Observations

This week established a lightweight Python-embedded DSL for expressing attention workloads declaratively. The DSL does not yet improve runtime or memory usage, but it introduces an important abstraction layer between workload specification and execution.

The main result is that attention workloads from different application scenarios can now be represented using a common high-level interface while still lowering correctly to trusted baseline PyTorch implementations. This supports the broader project goal of making attention optimization more programmable and scenario-aware rather than relying only on fixed-function kernels.

These results also respond directly to the project feedback. Instead of claiming a general-purpose optimization advantage over state-of-the-art kernels, the project now emphasizes customizable attention execution for settings such as multi-tenant serving, streaming inference, and long-context systems.

The next step will be to replace the current lowering path with tiled and memory-efficient transformations while preserving the same DSL interface.

## Week 2 Deliverables Completed

- Reused the Week 1 baseline attention implementations
- Defined a lightweight Python-embedded DSL for attention workloads
- Represented workload metadata for multi-tenant, streaming, and long-context scenarios
- Implemented lowering from DSL specifications to baseline PyTorch execution
- Verified correctness against direct reference implementations
- Established the interface that later weeks will optimize with tiled transformations